# Investment Research Multi-Agent System — Complete Standalone Notebook — Bounded Evidence

This notebook contains **all 17 source implementations as real, editable, executable code cells** followed by the seven workflow stages. Functions and classes are defined directly in the notebook kernel.

Run cells top-to-bottom. Outputs are saved in `data/raw/` and `data/processed/` relative to your Jupyter working directory. API access and the LLM need valid environment credentials; cached files are reused where supported. 

## A. Complete source code (17 files; executable notebook cells)

In [1]:
# Optional, run once in this notebook kernel if libraries are missing:
# %pip install pandas numpy requests python-dotenv openai pydantic yfinance ipython

import os
from pathlib import Path
from dotenv import load_dotenv
PROJECT_ROOT = Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / '.env', override=False)
for folder in ('raw','processed','cache'):
    (PROJECT_ROOT / 'data' / folder).mkdir(parents=True,exist_ok=True)
print('Working directory:', PROJECT_ROOT)


Working directory: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System


### Source implementation: `shared.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [2]:
import os
import random
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

SEED = 42
PROJECT_ROOT = Path.cwd().resolve()

PATHS = {
    "root": PROJECT_ROOT,
    "raw": PROJECT_ROOT / "data" / "raw",
    "processed": PROJECT_ROOT / "data" / "processed",
    "cache": PROJECT_ROOT / "data" / "cache",
    "docs": PROJECT_ROOT / "docs",
}


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def ensure_directories() -> None:
    for path in PATHS.values():
        path.mkdir(parents=True, exist_ok=True)


def load_project_env() -> None:
    load_dotenv(PROJECT_ROOT / ".env")


def get_target_tickers() -> list[str]:
    load_project_env()
    raw = os.getenv("TARGET_TICKERS", "AAPL")
    return [ticker.strip().upper() for ticker in raw.split(",") if ticker.strip()]


### Source implementation: `sec_ingestion.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [3]:
from __future__ import annotations

import time
import xml.etree.ElementTree as ET
from typing import Any

import pandas as pd
import requests

TICKER_MAP_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
ARCHIVES_BASE_URL = "https://www.sec.gov/Archives/edgar/data"

OUTPUT_COLUMNS = [
    "ticker",
    "company_name",
    "owner_name",
    "owner_title",
    "transaction_date",
    "filing_date",
    "transaction_code",
    "acquired_disposed",
    "shares",
    "price",
    "accession_number",
    "source_url",
]


def _session(user_agent: str) -> requests.Session:
    """Create an SEC-friendly HTTP session."""
    if not user_agent or "@" not in user_agent:
        raise ValueError(
            "SEC_USER_AGENT must identify the application and include a contact email."
        )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "www.sec.gov",
        }
    )
    return session


def _get_json(
    session: requests.Session,
    url: str,
    sleep_seconds: float,
) -> dict[str, Any]:
    """GET a JSON resource while respecting a small delay between SEC requests."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    time.sleep(sleep_seconds)
    return response.json()


def _get_text(
    session: requests.Session,
    url: str,
    sleep_seconds: float,
) -> str:
    """GET a text/XML resource while respecting a small SEC request delay."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    time.sleep(sleep_seconds)
    return response.text


def get_ticker_cik_map(
    user_agent: str,
    sleep_seconds: float = 0.12,
) -> dict[str, str]:
    """Return a mapping like {'AAPL': '0000320193'}."""
    session = _session(user_agent)

    # company_tickers.json is served from www.sec.gov.
    payload = _get_json(session, TICKER_MAP_URL, sleep_seconds)

    return {
        item["ticker"].upper(): str(item["cik_str"]).zfill(10)
        for item in payload.values()
    }


def _issuer_submissions(
    cik: str,
    user_agent: str,
    sleep_seconds: float,
) -> dict[str, Any]:
    """Load the SEC submissions JSON for an issuer."""
    session = _session(user_agent)

    # data.sec.gov uses the same SEC User-Agent policy but a different host.
    session.headers.pop("Host", None)

    url = SUBMISSIONS_URL.format(cik=cik)
    return _get_json(session, url, sleep_seconds)


def _recent_form4_filings(
    submissions: dict[str, Any],
    limit: int,
) -> list[dict[str, str]]:
    """Extract recent Form 4 / Form 4-A metadata from submissions JSON."""
    recent = submissions.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    accessions = recent.get("accessionNumber", [])
    filing_dates = recent.get("filingDate", [])

    filings: list[dict[str, str]] = []

    for form, accession, filing_date in zip(
        forms,
        accessions,
        filing_dates,
        strict=False,
    ):
        if form not in {"4", "4/A"}:
            continue

        filings.append(
            {
                "form": form,
                "accession_number": accession,
                "filing_date": filing_date,
            }
        )

        if len(filings) >= limit:
            break

    return filings


def _filing_directory_url(cik: str, accession_number: str) -> str:
    """Return the SEC Archives directory URL for a filing."""
    cik_no_leading_zeros = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{ARCHIVES_BASE_URL}/"
        f"{cik_no_leading_zeros}/"
        f"{accession_no_dashes}"
    )


def _find_ownership_xml_url(
    session: requests.Session,
    cik: str,
    accession_number: str,
    sleep_seconds: float,
) -> str:
    """Find the raw ownership XML file from a filing's index.json."""
    directory_url = _filing_directory_url(cik, accession_number)
    index_url = f"{directory_url}/index.json"

    payload = _get_json(session, index_url, sleep_seconds)
    items = payload.get("directory", {}).get("item", [])

    names = [
        item.get("name", "")
        for item in items
        if item.get("name")
    ]

    # Most ownership filings use form4.xml. Prefer it explicitly.
    preferred_names = [
        name
        for name in names
        if name.lower() == "form4.xml"
    ]

    if preferred_names:
        return f"{directory_url}/{preferred_names[0]}"

    # Fallback: use an XML file that is not an index/XSD file.
    xml_candidates = [
        name
        for name in names
        if name.lower().endswith(".xml")
        and "index" not in name.lower()
        and not name.lower().endswith(".xsd")
    ]

    if not xml_candidates:
        raise ValueError(
            f"No ownership XML found for accession {accession_number}. "
            f"Files in filing directory: {names}"
        )

    return f"{directory_url}/{xml_candidates[0]}"


def _strip_namespaces(root: ET.Element) -> None:
    """Remove XML namespaces so tag lookup is consistent."""
    for element in root.iter():
        if "}" in element.tag:
            element.tag = element.tag.split("}", 1)[1]


def _text(
    parent: ET.Element | None,
    path: str,
) -> str | None:
    """Safely return stripped text from a nested XML path."""
    if parent is None:
        return None

    element = parent.find(path)
    if element is None or element.text is None:
        return None

    value = element.text.strip()
    return value if value else None


def _number(
    parent: ET.Element | None,
    path: str,
) -> float | None:
    """Safely parse a numeric value from XML."""
    value = _text(parent, path)

    if value is None:
        return None

    try:
        return float(value.replace(",", ""))
    except ValueError:
        return None


def _owner_title(owner: ET.Element) -> str | None:
    """Create a readable owner role/title from Form 4 relationship fields."""
    relationship = owner.find("reportingOwnerRelationship")
    if relationship is None:
        return None

    officer_title = _text(relationship, "officerTitle")
    if officer_title:
        return officer_title

    roles: list[str] = []

    role_fields = [
        ("isDirector", "Director"),
        ("isOfficer", "Officer"),
        ("isTenPercentOwner", "10% Owner"),
        ("isOther", "Other"),
    ]

    for field, label in role_fields:
        value = _text(relationship, field)
        if value in {"1", "true", "TRUE"}:
            roles.append(label)

    return ", ".join(roles) if roles else None


def _transaction_rows(
    root: ET.Element,
    ticker: str,
    company_name: str | None,
    filing_date: str,
    accession_number: str,
    source_url: str,
) -> list[dict[str, Any]]:
    """Flatten ownership XML transactions into row dictionaries."""
    owners = root.findall("reportingOwner")

    owner_records: list[tuple[str | None, str | None]] = []
    for owner in owners:
        owner_name = _text(
            owner,
            "reportingOwnerId/rptOwnerName",
        )
        owner_records.append(
            (
                owner_name,
                _owner_title(owner),
            )
        )

    if not owner_records:
        owner_records = [(None, None)]

    transactions: list[ET.Element] = []
    transactions.extend(
        root.findall(
            "nonDerivativeTable/nonDerivativeTransaction"
        )
    )
    transactions.extend(
        root.findall(
            "derivativeTable/derivativeTransaction"
        )
    )

    rows: list[dict[str, Any]] = []

    for transaction in transactions:
        transaction_date = _text(
            transaction,
            "transactionDate/value",
        )
        transaction_code = _text(
            transaction,
            "transactionCoding/transactionCode",
        )
        shares = _number(
            transaction,
            "transactionAmounts/transactionShares/value",
        )
        price = _number(
            transaction,
            "transactionAmounts/transactionPricePerShare/value",
        )
        acquired_disposed = _text(
            transaction,
            "transactionAmounts/transactionAcquiredDisposedCode/value",
        )

        for owner_name, owner_title in owner_records:
            rows.append(
                {
                    "ticker": ticker,
                    "company_name": company_name,
                    "owner_name": owner_name,
                    "owner_title": owner_title,
                    "transaction_date": transaction_date,
                    "filing_date": filing_date,
                    "transaction_code": transaction_code,
                    "acquired_disposed": acquired_disposed,
                    "shares": shares,
                    "price": price,
                    "accession_number": accession_number,
                    "source_url": source_url,
                }
            )

    return rows


def _parse_ownership_xml(
    xml_text: str,
    ticker: str,
    fallback_company_name: str | None,
    filing_date: str,
    accession_number: str,
    source_url: str,
) -> list[dict[str, Any]]:
    """Parse one raw Form 4 ownership XML document."""
    root = ET.fromstring(xml_text)
    _strip_namespaces(root)

    if root.tag != "ownershipDocument":
        raise ValueError(
            f"Expected ownershipDocument root, received {root.tag!r}"
        )

    company_name = (
        _text(root, "issuer/issuerName")
        or fallback_company_name
    )

    return _transaction_rows(
        root=root,
        ticker=ticker,
        company_name=company_name,
        filing_date=filing_date,
        accession_number=accession_number,
        source_url=source_url,
    )


def fetch_form4_transactions(
    tickers: list[str],
    user_agent: str,
    filings_per_ticker: int = 20,
    sleep_seconds: float = 0.12,
) -> pd.DataFrame:
    """Fetch recent SEC Form 4 transactions for one or more tickers."""
    ticker_map = get_ticker_cik_map(
        user_agent=user_agent,
        sleep_seconds=sleep_seconds,
    )

    archive_session = _session(user_agent)
    all_rows: list[dict[str, Any]] = []

    for raw_ticker in tickers:
        ticker = raw_ticker.strip().upper()

        cik = ticker_map.get(ticker)
        if cik is None:
            print(f"SEC: ticker not found in SEC ticker map: {ticker}")
            continue

        submissions = _issuer_submissions(
            cik=cik,
            user_agent=user_agent,
            sleep_seconds=sleep_seconds,
        )

        company_name = submissions.get("name")

        filings = _recent_form4_filings(
            submissions=submissions,
            limit=filings_per_ticker,
        )

        print(
            f"SEC: {ticker} | CIK {cik} | "
            f"{len(filings)} recent Form 4 filing(s)"
        )

        for filing in filings:
            accession_number = filing["accession_number"]

            try:
                xml_url = _find_ownership_xml_url(
                    session=archive_session,
                    cik=cik,
                    accession_number=accession_number,
                    sleep_seconds=sleep_seconds,
                )

                xml_text = _get_text(
                    archive_session,
                    xml_url,
                    sleep_seconds,
                )

                rows = _parse_ownership_xml(
                    xml_text=xml_text,
                    ticker=ticker,
                    fallback_company_name=company_name,
                    filing_date=filing["filing_date"],
                    accession_number=accession_number,
                    source_url=xml_url,
                )

                all_rows.extend(rows)

                print(
                    f"SEC: parsed {ticker} {accession_number} "
                    f"({len(rows)} transaction row(s))"
                )

            except (
                requests.RequestException,
                ET.ParseError,
                ValueError,
            ) as exc:
                print(
                    f"SEC: failed {ticker} {accession_number}: {exc}"
                )

    if not all_rows:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    result = pd.DataFrame(all_rows)

    for column in OUTPUT_COLUMNS:
        if column not in result.columns:
            result[column] = pd.NA

    return result[OUTPUT_COLUMNS].reset_index(drop=True)


### Source implementation: `alpha_vantage_ingestion.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [4]:
from __future__ import annotations

from typing import Any

import pandas as pd
import requests

ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"


def _alpha_vantage_request(params: dict[str, Any], api_key: str) -> dict[str, Any]:
    response = requests.get(
        ALPHA_VANTAGE_BASE_URL,
        params={**params, "apikey": api_key},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    for key in ("Error Message", "Information", "Note"):
        if key in payload:
            raise RuntimeError(payload[key])

    return payload


def fetch_daily_market_data(
    tickers: list[str],
    api_key: str,
) -> pd.DataFrame:
    """Fetch compact daily OHLCV data."""
    rows: list[dict[str, Any]] = []

    for ticker in tickers:
        symbol = ticker.strip().upper()

        payload = _alpha_vantage_request(
            {
                "function": "TIME_SERIES_DAILY",
                "symbol": symbol,
                "outputsize": "compact",
            },
            api_key,
        )

        for date, values in payload.get("Time Series (Daily)", {}).items():
            rows.append(
                {
                    "ticker": symbol,
                    "date": date,
                    "open": values.get("1. open"),
                    "high": values.get("2. high"),
                    "low": values.get("3. low"),
                    "close": values.get("4. close"),
                    "volume": values.get("5. volume"),
                }
            )

    return pd.DataFrame(rows)


### Source implementation: `marketaux_ingestion.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [5]:
"""Marketaux financial-news ingestion helpers."""

from __future__ import annotations

import math
import time
from typing import Any

import pandas as pd
import requests

MARKETAUX_BASE_URL = "https://api.marketaux.com/v1/news/all"


def _marketaux_request(params: dict[str, Any], api_token: str) -> dict[str, Any]:
    response = requests.get(
        MARKETAUX_BASE_URL,
        params={**params, "api_token": api_token},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    meta = payload.get("meta", {})
    if meta.get("code"):
        raise RuntimeError(
            meta.get("message")
            or f"Marketaux error: {meta.get('code')}"
        )

    return payload


def fetch_financial_news(
    tickers: list[str],
    api_token: str,
    limit_per_ticker: int = 9,
    page_size: int = 3,
    pause_seconds: float = 0.5,
) -> pd.DataFrame:
    """Fetch recent financial news with pagination."""
    rows: list[dict[str, Any]] = []

    for ticker in tickers:
        symbol = ticker.strip().upper()
        ticker_rows: list[dict[str, Any]] = []
        pages_needed = math.ceil(limit_per_ticker / page_size)

        for page in range(1, pages_needed + 1):
            remaining = limit_per_ticker - len(ticker_rows)
            if remaining <= 0:
                break

            payload = _marketaux_request(
                {
                    "symbols": symbol,
                    "filter_entities": "true",
                    "language": "en",
                    "limit": min(page_size, remaining),
                    "page": page,
                },
                api_token,
            )

            articles = payload.get("data", [])
            if not articles:
                break

            for article in articles:
                ticker_rows.append(
                    {
                        "ticker": symbol,
                        "published_at": article.get("published_at"),
                        "title": article.get("title"),
                        "description": article.get("description"),
                        "content": (
                            article.get("snippet")
                            or article.get("description")
                        ),
                        "source": article.get("source"),
                        "url": article.get("url"),
                    }
                )

                if len(ticker_rows) >= limit_per_ticker:
                    break

            if page < pages_needed:
                time.sleep(pause_seconds)

        rows.extend(ticker_rows[:limit_per_ticker])

    return pd.DataFrame(rows)


### Source implementation: `evidence.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [6]:
from __future__ import annotations

import pandas as pd


def assemble_daily_evidence(
    market: pd.DataFrame,
    news: pd.DataFrame,
    sec: pd.DataFrame,
) -> pd.DataFrame:
    """Create one ticker/date evidence table without making investment judgments."""
    market = market.copy()
    news = news.copy()
    sec = sec.copy()

    market["date"] = pd.to_datetime(market["date"], utc=True).dt.normalize()
    news["date"] = pd.to_datetime(news["published_at"], utc=True).dt.normalize()
    sec["date"] = pd.to_datetime(sec["transaction_date"], utc=True).dt.normalize()

    news_daily = (
        news.groupby(["ticker", "date"], dropna=False)
        .agg(
            news_count=("title", "size"),
            news_titles=("title", lambda values: " || ".join(values.dropna().astype(str).head(10))),
            news_urls=("url", lambda values: " || ".join(values.dropna().astype(str).head(10))),
        )
        .reset_index()
    )

    sec_daily = (
        sec.groupby(["ticker", "date"], dropna=False)
        .agg(
            insider_transaction_count=("owner_name", "size"),
            insider_total_value=("transaction_value", "sum"),
            insider_names=(
                "owner_name",
                lambda values: " || ".join(
                    values.dropna().astype(str).unique()[:10]
                ),
            ),
        )
        .reset_index()
    )

    evidence = market.merge(news_daily, on=["ticker", "date"], how="outer")
    evidence = evidence.merge(sec_daily, on=["ticker", "date"], how="outer")
    evidence["news_count"] = evidence["news_count"].fillna(0).astype("int64")

    if "adj_close" in evidence and evidence["adj_close"].isna().all():
        evidence = evidence.drop(columns="adj_close")

    evidence = evidence.sort_values(["ticker", "date"]).reset_index(drop=True)
    return evidence


### Source implementation: `memory.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [7]:
import json
import os
import datetime
from typing import Dict, List, Any, Optional


class ResearchMemoryStore:
    """
    Manages persistent, cross-run memory for stock research analyses.
    Stores historical evaluation scores, recurring critique notes, and key takeaways per ticker.
    """

    def __init__(self, filepath: str = "data/processed/memory_store.json"):
        self.filepath = filepath
        self._ensure_dir_exists()
        self.memory: Dict[str, List[Dict[str, Any]]] = self._load_memory()

    def _ensure_dir_exists(self) -> None:
        """Ensures the destination folder exists before reading or writing."""
        dir_name = os.path.dirname(self.filepath)
        if dir_name and not os.path.exists(dir_name):
            os.makedirs(dir_name, exist_ok=True)

    def _load_memory(self) -> Dict[str, List[Dict[str, Any]]]:
        """Loads memory data from JSON file if present."""
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError) as e:
                print(f"[MemoryStore] Warning: Failed to load memory file ({e}). Starting fresh.")
                return {}
        return {}

    def _save_memory(self) -> None:
        """Persists memory data to JSON file."""
        try:
            with open(self.filepath, "w", encoding="utf-8") as f:
                json.dump(self.memory, f, indent=2, ensure_ascii=False)
        except OSError as e:
            print(f"[MemoryStore] Error saving memory to {self.filepath}: {e}")

    def get_ticker_memory(self, ticker: str, max_entries: int = 3) -> str:
        """
        Fetches formatted history for a given ticker to inject into prompts.
        
        Parameters:
            ticker (str): Stock symbol (e.g., 'AAPL', 'NVDA').
            max_entries (int): Number of most recent runs to include.
            
        Returns:
            str: Formatted memory text block for LLM context injection.
        """
        symbol = ticker.upper()
        ticker_history = self.memory.get(symbol, [])

        if not ticker_history:
            return f"No previous research memory recorded for {symbol}."

        # Take the N most recent entries
        recent_runs = ticker_history[-max_entries:]
        memory_lines = [f"=== HISTORICAL MEMORY FOR {symbol} ==="]

        for i, entry in enumerate(recent_runs, 1):
            timestamp = entry.get("timestamp", "Unknown Date")
            score = entry.get("score", "N/A")
            summary = entry.get("summary", "No summary provided.")
            lessons = entry.get("lessons_learned", "")

            line = f"Run #{i} [{timestamp}] - Final Quality Score: {score}/10\n  Summary: {summary}"
            if lessons:
                line += f"\n  Lessons Learned: {lessons}"
            memory_lines.append(line)

        return "\n".join(memory_lines)

    def save_run(
        self,
        ticker: str,
        timestamp: Optional[str],
        score: int,
        summary: str,
        lessons_learned: Optional[str] = None
    ) -> None:
        """
        Appends a completed research run record to the memory store.
        
        Parameters:
            ticker (str): Stock symbol.
            timestamp (str): Execution timestamp string.
            score (int): Final evaluation score (1-10).
            summary (str): Brief summary of run outcome and evaluator feedback.
            lessons_learned (str): Optional recurring critique or pitfall note to avoid next time.
        """
        symbol = ticker.upper()
        if symbol not in self.memory:
            self.memory[symbol] = []

        if not timestamp:
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        entry = {
            "timestamp": timestamp,
            "score": score,
            "summary": summary,
            "lessons_learned": lessons_learned or ""
        }

        self.memory[symbol].append(entry)
        self._save_memory()
        print(f"[MemoryStore] Saved run for {symbol} to {self.filepath}.")

    def clear_memory(self, ticker: Optional[str] = None) -> None:
        """Clears memory for a specific ticker or wipes the entire memory store."""
        if ticker:
            symbol = ticker.upper()
            if symbol in self.memory:
                del self.memory[symbol]
                self._save_memory()
                print(f"[MemoryStore] Cleared memory for {symbol}.")
        else:
            self.memory = {}
            self._save_memory()
            print("[MemoryStore] Cleared all memory records.")

### Source implementation: `market_agent.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [8]:

"""Market specialist agent."""

from __future__ import annotations

from pathlib import Path

import pandas as pd


def analyze_market(ticker: str, project_root: str | Path = ".") -> dict:
    """Analyze processed market evidence for a ticker."""

    ticker = ticker.upper().strip()
    project_root = Path(project_root)

    market_path = (
        project_root
        / "data"
        / "processed"
        / "market_data_clean.csv"
    )

    if not market_path.exists():
        raise FileNotFoundError(
            f"Market data file not found: {market_path}"
        )

    market = pd.read_csv(market_path)

    ticker_market = market[
        market["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_market.empty:
        return {
            "agent": "market_agent",
            "ticker": ticker,
            "row_count": 0,
            "message": f"No market data found for {ticker}.",
        }

    ticker_market["date"] = pd.to_datetime(
        ticker_market["date"],
        errors="coerce",
        utc=True,
    )

    ticker_market = ticker_market.sort_values("date")

    latest = ticker_market.iloc[-1]

    average_return = ticker_market["daily_return"].mean()
    average_volume = ticker_market["volume"].mean()

    return {
        "agent": "market_agent",
        "ticker": ticker,
        "row_count": len(ticker_market),
        "start_date": ticker_market["date"].min(),
        "end_date": ticker_market["date"].max(),
        "latest_close": float(latest["close"]),
        "latest_volume": float(latest["volume"]),
        "average_daily_return": float(average_return),
        "average_volume": float(average_volume),
    }


### Source implementation: `news_agent.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [9]:

"""News specialist agent."""

from __future__ import annotations

from pathlib import Path

import pandas as pd


def analyze_news(ticker: str, project_root: str | Path = ".") -> dict:
    """Analyze processed news-research results for a ticker."""

    ticker = ticker.upper().strip()
    project_root = Path(project_root)

    news_path = (
        project_root
        / "data"
        / "processed"
        / "news_research_results.csv"
    )

    if not news_path.exists():
        raise FileNotFoundError(
            f"News research file not found: {news_path}"
        )

    news = pd.read_csv(news_path)

    ticker_news = news[
        news["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_news.empty:
        return {
            "agent": "news_agent",
            "ticker": ticker,
            "article_count": 0,
            "categories": {},
            "research_notes": [],
            "message": f"No processed news found for {ticker}.",
        }

    category_counts = (
        ticker_news["category"]
        .value_counts()
        .to_dict()
    )

    research_notes = ticker_news[
        [
            "title",
            "category",
            "event",
            "summary",
            "url",
        ]
    ].to_dict(orient="records")

    return {
        "agent": "news_agent",
        "ticker": ticker,
        "article_count": len(ticker_news),
        "categories": category_counts,
        "research_notes": research_notes,
    }


### Source implementation: `insider_agent.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [10]:

"""Insider-transaction specialist agent."""

from __future__ import annotations

from pathlib import Path

import pandas as pd


def analyze_insider_activity(
    ticker: str,
    project_root: str | Path = ".",
) -> dict:
    """Analyze SEC Form 4 insider transactions for a ticker."""

    ticker = ticker.upper().strip()
    project_root = Path(project_root)

    insider_path = (
        project_root
        / "data"
        / "processed"
        / "sec_form4_clean.csv"
    )

    if not insider_path.exists():
        raise FileNotFoundError(
            f"SEC Form 4 file not found: {insider_path}"
        )

    insider = pd.read_csv(insider_path)

    ticker_insider = insider[
        insider["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_insider.empty:
        return {
            "agent": "insider_agent",
            "ticker": ticker,
            "transaction_count": 0,
            "message": f"No insider transactions found for {ticker}.",
        }

    ticker_insider["transaction_date"] = pd.to_datetime(
        ticker_insider["transaction_date"],
        errors="coerce",
        utc=True,
    )

    ticker_insider["transaction_value"] = pd.to_numeric(
        ticker_insider["transaction_value"],
        errors="coerce",
    )

    ticker_insider["shares"] = pd.to_numeric(
        ticker_insider["shares"],
        errors="coerce",
    )

    acquired = ticker_insider[
        ticker_insider["acquired_disposed"] == "A"
    ]

    disposed = ticker_insider[
        ticker_insider["acquired_disposed"] == "D"
    ]

    return {
        "agent": "insider_agent",
        "ticker": ticker,
        "transaction_count": len(ticker_insider),
        "unique_insiders": ticker_insider["owner_name"].nunique(),
        "acquired_transactions": len(acquired),
        "disposed_transactions": len(disposed),
        "total_transaction_value": float(
            ticker_insider["transaction_value"].fillna(0).sum()
        ),
        "start_date": ticker_insider["transaction_date"].min(),
        "end_date": ticker_insider["transaction_date"].max(),
    }


### Source implementation: `planner.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [11]:
"""Planner agent: checks what data we have for a ticker, then builds the research steps."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

EVIDENCE_FILES = {
    "news": "news_research_results.csv",
    "market": "market_data_clean.csv",
    "insider": "sec_form4_clean.csv",
}


def check_available_evidence(ticker: str, project_root: str | Path = ".") -> dict:
    """Count how many rows each processed dataset has for this ticker."""

    ticker = ticker.upper().strip()
    processed = Path(project_root) / "data" / "processed"

    counts = {}
    news_categories = {}

    for source, filename in EVIDENCE_FILES.items():
        path = processed / filename

        if not path.exists():
            counts[source] = 0
            continue

        data = pd.read_csv(path)
        rows = data[data["ticker"].astype(str).str.upper() == ticker]
        counts[source] = len(rows)

        if source == "news" and not rows.empty and "category" in rows.columns:
            news_categories = rows["category"].value_counts().to_dict()

    return {
        "ticker": ticker,
        "counts": counts,
        "news_categories": news_categories,
    }


def _default_evidence(ticker: str) -> dict:
    """Used when we don't check the data: treat every source as available."""

    return {
        "ticker": ticker,
        "counts": {"news": None, "market": None, "insider": None},
        "news_categories": {},
    }


def generate_research_plan(ticker: str, project_root: str | Path | None = None) -> dict:
    """Build the research plan for a ticker.

    If project_root is given, a step is only added when that source has data.
    If not, it returns the full 6-step plan.
    """

    ticker = ticker.upper().strip()

    if project_root is None:
        evidence = _default_evidence(ticker)
    else:
        evidence = check_available_evidence(ticker, project_root)

    counts = evidence["counts"]
    categories = evidence["news_categories"]

    candidate_steps = []
    decisions = []

    # News step
    if counts["news"] == 0:
        decisions.append(
            {"task": "news_analysis", "included": False,
             "reason": f"No processed news articles found for {ticker}."}
        )
    else:
        description = f"Review recent financial news and macro developments relevant to {ticker}."
        reason = "News data available."
        if counts["news"]:
            top_category = max(categories, key=categories.get) if categories else None
            reason = f"Found {counts['news']} processed news articles."
            if top_category:
                top_count = categories[top_category]
                description += f" Most coverage is {top_category} ({top_count} articles)."
        candidate_steps.append(("news_analysis", description))
        decisions.append({"task": "news_analysis", "included": True, "reason": reason})

    # Market step
    if counts["market"] == 0:
        decisions.append(
            {"task": "market_analysis", "included": False,
             "reason": f"No market data found for {ticker}."}
        )
    else:
        reason = (
            f"Found {counts['market']} trading days of market data."
            if counts["market"] else "Market data available."
        )
        candidate_steps.append(
            ("market_analysis",
             f"Analyze recent price, return, and trading-volume behavior for {ticker}.")
        )
        decisions.append({"task": "market_analysis", "included": True, "reason": reason})

    # Insider step
    if counts["insider"] == 0:
        decisions.append(
            {"task": "insider_analysis", "included": False,
             "reason": f"No SEC Form 4 insider transactions found for {ticker}."}
        )
    else:
        reason = (
            f"Found {counts['insider']} SEC Form 4 insider transactions."
            if counts["insider"] else "Insider data available."
        )
        candidate_steps.append(
            ("insider_analysis", f"Review recent SEC Form 4 insider transactions for {ticker}.")
        )
        decisions.append({"task": "insider_analysis", "included": True, "reason": reason})

    # risk, catalyst and summary steps only make sense if we have some data
    if candidate_steps:
        candidate_steps.extend(
            [
                ("risk_analysis",
                 f"Identify important risks using the available evidence for {ticker}."),
                ("catalyst_analysis",
                 f"Identify possible catalysts or notable events affecting {ticker}."),
                ("final_synthesis",
                 f"Synthesize the research evidence into a concise investment-research "
                 f"summary for {ticker}."),
            ]
        )
    else:
        decisions.append(
            {"task": "final_synthesis", "included": False,
             "reason": f"No evidence available for {ticker}, so there is nothing to synthesize."}
        )

    research_steps = [
        {"step_id": step_id, "task": task, "description": description}
        for step_id, (task, description) in enumerate(candidate_steps, start=1)
    ]

    return {
        "ticker": ticker,
        "objective": f"Conduct structured multi-agent investment research for {ticker}.",
        "evidence_checked": project_root is not None,
        "evidence_counts": counts,
        "research_steps": research_steps,
        "planning_decisions": decisions,
    }


### Source implementation: `router.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [12]:

"""Router Agent for the investment research multi-agent system."""

from __future__ import annotations


def route_step(step: dict) -> dict:
    """Route a planner step to the appropriate specialist agent."""

    task = step.get("task", "").strip().lower()

    routing_map = {
        "news_analysis": "news_agent",
        "market_analysis": "market_agent",
        "insider_analysis": "insider_agent",
        "risk_analysis": "synthesis_agent",
        "catalyst_analysis": "synthesis_agent",
        "final_synthesis": "synthesis_agent",
    }

    selected_agent = routing_map.get(task, "synthesis_agent")

    reasons = {
        "news_agent": "This task depends primarily on processed financial-news evidence.",
        "market_agent": "This task depends primarily on market price, return, and volume evidence.",
        "insider_agent": "This task depends primarily on SEC Form 4 insider-transaction evidence.",
        "synthesis_agent": "This task requires combining evidence from multiple sources.",
    }

    return {
        "step_id": step.get("step_id"),
        "task": task,
        "agent": selected_agent,
        "reason": reasons[selected_agent],
        "description": step.get("description", ""),
    }


### Source implementation: `synthesis_agent.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [13]:

"""Synthesis specialist for combining multi-source research evidence."""

from __future__ import annotations


def synthesize_research(
    ticker: str,
    news_result: dict,
    market_result: dict,
    insider_result: dict,
) -> dict:
    """Combine specialist outputs into structured research conclusions."""

    ticker = ticker.upper().strip()

    risks = []
    catalysts = []

    # News-derived signals
    categories = news_result.get("categories", {})

    if categories.get("macro_market", 0) > 0:
        risks.append(
            "Macro or broader market developments may affect the stock."
        )

    if categories.get("analyst_investor", 0) > 0:
        catalysts.append(
            "Recent analyst or investor commentary may influence market expectations."
        )

    # Market-derived signals
    avg_return = market_result.get("average_daily_return")

    if avg_return is not None:
        if avg_return < 0:
            risks.append(
                "Average daily market return over the available period is negative."
            )
        else:
            catalysts.append(
                "Average daily market return over the available period is positive."
            )

    # Insider-derived signals
    acquired = insider_result.get("acquired_transactions", 0)
    disposed = insider_result.get("disposed_transactions", 0)

    if disposed > acquired:
        risks.append(
            "Disposed insider transactions outnumber acquired transactions in the available Form 4 data."
        )

    if acquired > 0:
        catalysts.append(
            "The available Form 4 data includes insider acquisition transactions."
        )

    return {
        "agent": "synthesis_agent",
        "ticker": ticker,
        "risks": risks,
        "catalysts": catalysts,
        "summary": {
            "news_articles": news_result.get("article_count", 0),
            "market_rows": market_result.get("row_count", 0),
            "insider_transactions": insider_result.get(
                "transaction_count", 0
            ),
        },
    }


### Source implementation: `orchestrator.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [14]:
"""Orchestrator for the multi-agent investment research workflow."""

from __future__ import annotations

from pathlib import Path

# the function each specialist agent runs
SPECIALIST_TOOLS = {
    "news_agent": analyze_news,
    "market_agent": analyze_market,
    "insider_agent": analyze_insider_activity,
}

SKIPPED_MESSAGES = {
    "news_agent": ("article_count", "No processed news found for {ticker} (skipped by planner)."),
    "market_agent": ("row_count", "No market data found for {ticker} (skipped by planner)."),
    "insider_agent": (
        "transaction_count",
        "No insider transactions found for {ticker} (skipped by planner).",
    ),
}


def orchestrate_research(
    ticker: str,
    project_root: str | Path = ".",
) -> dict:
    """Run the planner, router, specialists and synthesis for one ticker.

    Only the agents that the router picked get called.
    """

    ticker = ticker.upper().strip()
    project_root = Path(project_root)

    plan = generate_research_plan(ticker, project_root)

    routing_decisions = [route_step(step) for step in plan["research_steps"]]
    selected_agents = {decision["agent"] for decision in routing_decisions}

    # only call the agents that got a step, the rest are marked as skipped
    specialist_results = {}
    for agent_name, tool in SPECIALIST_TOOLS.items():
        if agent_name in selected_agents:
            specialist_results[agent_name] = tool(ticker, project_root)
        else:
            count_key, message = SKIPPED_MESSAGES[agent_name]
            specialist_results[agent_name] = {
                "agent": agent_name,
                "ticker": ticker,
                count_key: 0,
                "skipped": True,
                "message": message.format(ticker=ticker),
            }

    synthesis_result = synthesize_research(
        ticker,
        specialist_results["news_agent"],
        specialist_results["market_agent"],
        specialist_results["insider_agent"],
    )
    specialist_results["synthesis_agent"] = synthesis_result

    executed_steps = [
        {
            "step_id": decision["step_id"],
            "task": decision["task"],
            "agent": decision["agent"],
            "reason": decision["reason"],
            "result": specialist_results[decision["agent"]],
        }
        for decision in routing_decisions
    ]

    return {
        "ticker": ticker,
        "plan": plan,
        "routing_decisions": routing_decisions,
        "agents_called": sorted(selected_agents & SPECIALIST_TOOLS.keys()),
        "executed_steps": executed_steps,
        "specialist_results": specialist_results,
        "final_synthesis": synthesis_result,
    }


def display_orchestration(result: dict) -> None:
    """Print a readable summary of the workflow."""

    print("=" * 72)
    print(f"Ticker: {result['ticker']}")
    print(f"Objective: {result['plan']['objective']}")
    print("=" * 72)

    print("Planner decisions:")
    for decision in result["plan"]["planning_decisions"]:
        status = "include" if decision["included"] else "skip"
        print(f"- {decision['task']}: {status} ({decision['reason']})")

    print("=" * 72)
    for step in result["executed_steps"]:
        print(f"Step {step['step_id']}: {step['task']} -> {step['agent']}")

    print(f"Agents called: {', '.join(result['agents_called']) or 'none'}")

    print("=" * 72)
    print("Final synthesis")
    print("Risks:")
    for risk in result["final_synthesis"]["risks"]:
        print(f"- {risk}")

    print("Catalysts:")
    for catalyst in result["final_synthesis"]["catalysts"]:
        print(f"- {catalyst}")


### Source implementation: `evaluator.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [15]:
import os
from typing import List
from pydantic import BaseModel, Field
from openai import OpenAI
import re

# Initialize client using environment variables configured in .env
_evaluator_api_key = os.getenv("LLM_API_KEY")
_evaluator_base_url = os.getenv("LLM_BASE_URL")

# If base_url is empty string or None, set to None so OpenAI uses its default URL
if not _evaluator_base_url:
    _evaluator_base_url = None

_evaluator_client = OpenAI(
    api_key=_evaluator_api_key or "NOT_SET_YET",
    base_url=_evaluator_base_url
)

EVALUATOR_MODEL_NAME = os.getenv("LLM_MODEL", "gpt-4o-mini")


# Pydantic schemas for evaluation rubric and results
class EvaluationRubric(BaseModel):
    factual_grounding_score: int = Field(default=0, description="Score (0-10) for factual alignment.")
    completeness_score: int = Field(default=0, description="Score (0-10) for required sections.")
    logical_coherence_score: int = Field(default=0, description="Score (0-10) for narrative consistency.")
    clarity_and_structure: int = Field(default=0, description="Score (0-10) for formatting and structure.")

class EvaluationResult(BaseModel):
    overall_score: int
    passed: bool
    rubric: EvaluationRubric
    failure_reason: str = Field(default="")
    refinement_feedback: str = Field(default="")
    missing_elements: List[str] = Field(default_factory=list)

# Evaluator prompts 
EVALUATOR_SYSTEM_PROMPT = """
You are a Strict Financial Research Evaluation Agent auditing automated investment reports.

CRITICAL EVALUATION RUBRIC:
1. HARD PASS/FAIL GATES (Immediate Failure if False):
   - NO_PLACEHOLDERS: Draft must contain zero bracketed placeholders (e.g., '[Insert Date]', '[TBD]').
   - MANDATORY_SECTIONS: Must contain explicit headers for Market Metrics, SEC Insider Activity, and News.
   - NO_FINANCIAL_ADVICE: Must not output explicit 'BUY/SELL' recommendations or fabricated target prices.

2. FACTUAL GROUNDING & PRECISION (0-10):
   - Deduct 2 points for every unverified figure, rounded dollar amount (when exact figures exist in evidence), or misaligned trade date.
   - Every metric must cite its source origin (e.g., [SEC Form 4], [Alpha Vantage], [Marketaux]).
   - Do not penalize for domain-specific tags as long as [Marketaux] is also listed as a source

3. COMPLETENESS (0-10):
   - All available ticker data points in Raw Evidence must be represented.
   - If a data category is absent from Raw Evidence, an explicit missing-data disclaimer must be present under that section.
   - If a data metric (e.g., quarterly revenue) is NOT present in the raw evidence dataset, award FULL POINTS if the draft contains an explicit disclaimer statement under that section header explaining that the data was absent from the raw evidence.
    - Do NOT deduct points for missing raw data if an explicit disclaimer is present.

4. LOGICAL COHERENCE (0-10):
   - Flag any narrative contradictions (e.g., claiming bullish sentiment when price and insider trades are heavily negative).

Pass Condition:
Draft passes ONLY if ALL Hard Gates == True AND Factual Grounding >= 8 AND Overall Score >= 8.
"""

EVALUATOR_USER_PROMPT_TEMPLATE = """
--- CANDIDATE RESEARCH DRAFT ---
{current_draft}

--- RAW EVIDENCE SOURCE ---
{raw_evidence}

Please evaluate the draft against the raw evidence and return a structured EvaluationResult.
"""


def evaluate_draft(
    current_draft: str,
    raw_evidence: str,
    model: str = EVALUATOR_MODEL_NAME
) -> EvaluationResult:
    """
    Evaluates a candidate research draft using programmatic hard gates first,
    followed by structured LLM evaluation if hard gates pass.
    
    Parameters:
        current_draft (str): The research report draft to evaluate.
        raw_evidence (str): Ground truth evidence (daily_evidence.csv / news_research_results.csv).
        model (str): The LLM model to execute evaluation.
        
    Returns:
        EvaluationResult: Structured result object containing scores, pass/fail status with reason, and critique.
    """
    # Step 1: Manual hard gate checks before LLM as a judge
    failures = []

    # Cheks for exact mandatory headers
    mandatory_headers = [
        "## Market Metrics",
        "## SEC Form 4 Insider Activity",
        "## Financial News Analysis",
        "## Financials & Revenue Performance"
    ]
    missing_headers = [h for h in mandatory_headers if h not in current_draft]
    if missing_headers:
        failures.append(f"Missing exact mandatory section headers: {missing_headers}")

    # Check for line by line source citations
    bullet_lines = [
        line.strip() 
        for line in current_draft.split("\n") 
        if line.strip().startswith("-") or line.strip().startswith("*")
    ]
    uncited_bullets = [
        line for line in bullet_lines 
        if "[Source:" not in line and "not recorded" not in line.lower()
    ]
    if uncited_bullets:
        failures.append(
            f"EVERY individual bullet point containing factual metrics MUST include an inline source tag "
            f"(e.g., [Source: Alpha Vantage], [Source: SEC Form 4], [Source: Marketaux]). "
            f"Missing on: {uncited_bullets[:2]}"
        )

    # Ensures no ellipsis are in the draft
    if "..." in current_draft:
        failures.append(
            "Draft contains literal ellipsis '...'. Write full sentences and use explicit disclaimers for missing data."
        )

    # Ensures no bracketed placeholders are in the draft
    placeholder_pattern = r"\\[(Insert|TBD|Date|Ticker|Header|Name|Value)[^\\]]*\]"
    found_placeholders = re.findall(placeholder_pattern, current_draft, re.IGNORECASE)
    if found_placeholders:
        failures.append(f"Draft contains template placeholders matching pattern: {found_placeholders}")

    # Break out of evaluation if any hard gates fail
    # Do not proceed with LLM as a judge if hard gates fail
    if failures:
        failure_msg = " | ".join(failures)
        return EvaluationResult(
            overall_score=4,
            passed=False,
            rubric=EvaluationRubric(
                factual_grounding_score=4,
                completeness_score=4,
                logical_coherence_score=4,
                clarity_and_structure=4
            ),
            failure_reason=f"Hard gate failure: {failure_msg}",
            refinement_feedback=f"Please fix the following issues: {failure_msg}",
            missing_elements=failures
        )

    # Step 2: LLM as a judge (only runs if hard gates pass)
    user_message = EVALUATOR_USER_PROMPT_TEMPLATE.format(
        current_draft=current_draft,
        raw_evidence=raw_evidence
    )

    # Generate structured response from LLM based on system_prompt
    response = _evaluator_client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        response_format=EvaluationResult,
        temperature=0.1,
        max_tokens=1500
    )

    eval_result: EvaluationResult = response.choices[0].message.parsed
    return eval_result

### Source implementation: `optimizer.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

**OpenRouter prompt-size fix:** The Optimizer now uses a bounded evidence sample rather than sending entire CSV datasets. The full data remains saved in `data/processed/` for inspection. If you need claims about older rows, expand the bounded evidence deliberately and verify the source data.

In [16]:
import os
from typing import List, Optional
from openai import OpenAI
from datetime import datetime

# Read env variables
_optimizer_api_key = os.getenv("LLM_API_KEY")
_optimizer_base_url = os.getenv("LLM_BASE_URL")

# If base_url is empty string or None, set to None so OpenAI uses its default URL
if not _optimizer_base_url:
    _optimizer_base_url = None

_optimizer_client = OpenAI(
    api_key=_optimizer_api_key or "NOT_SET_YET",
    base_url=_optimizer_base_url
)

OPTIMIZER_MODEL_NAME = os.getenv("LLM_MODEL", "gpt-4o-mini")

OPTIMIZER_SYSTEM_PROMPT = """
You are a Financial Research Optimization Agent.

CRITICAL CITATION & FORMATTING RULES:
- You MUST append an inline source tag (e.g., [Source: Alpha Vantage], [Source: SEC Form 4], [Source: Marketaux]) to EVERY SINGLE bullet point that contains data or metrics.
- Do NOT place a single tag at the end of a section. EVERY line must have its own tag!
- NEVER output trailing ellipsis '...' or incomplete sentences (e.g., 'opened at $3 ...').
- Use EXACT source names for citations:
   - Use [Source: Alpha Vantage] for market price/volume data.
   - Use [Source: SEC Form 4] for insider trading data.
   - Use [Source: Marketaux] for financial news articles.
   - Do NOT use generic tags like [Source: Daily Market Evidence].
- Every bullet point must be a complete, well-formed sentence.

Example Correct Format:
## Market Metrics
- Trading Days Analyzed: 22 [Source: Alpha Vantage]
- Latest Close Price: $515.31 [Source: Alpha Vantage]
- Average Daily Volume: 20,926,087 shares [Source: Alpha Vantage]
"""


OPTIMIZER_USER_PROMPT_TEMPLATE = """
You are refining a stock research report to pass a strict quality gate.

Current Draft:
{current_draft}

Evaluator Critique & Failure Reason:
- Main Failure Reason: {failure_reason}
- Missing Elements to Add: {missing_elements}
- Specific Refinement Feedback: {feedback}

Raw Evidence Source:
{raw_evidence}

Historical Memory for this Ticker: {memory_context}

Instructions:
1. Revise the current draft to directly integrate the missing elements listed above.
2. Ensure every added metric (prices, Form 4 trade values, news headlines) is strictly grounded in the Raw Evidence.
3. If a requested metric is not in the Raw Evidence, state its absence explicitly as instructed.
4. Use historical memory to avoid repeating past issues. 
"""


def optimize_draft(
    current_draft: str,
    feedback: str,
    missing_elements: List[str],
    raw_evidence: str,
    overall_score: int = 0,
    memory_context: Optional[str] = None,
    failure_reason: str = None,
    model: str = OPTIMIZER_MODEL_NAME
) -> str:
    """
    Refines a candidate research draft using feedback and missing facts from the Evaluator.
    
    Parameters:
        current_draft (str): The candidate research draft to improve.
        feedback (str): Constructive critique instructions from EvaluationResult.
        missing_elements (List[str]): List of missing items identified by the Evaluator.
        raw_evidence (str): Ground truth data (from daily_evidence.csv / news_research_results.csv).
        overall_score (int): Previous quality score.
        memory_context (str): Historical run memory/notes for this ticker.
        model (str): LLM model identifier.
        
    Returns:
        str: The updated, higher-quality research report draft.
    """
    formatted_missing = "\n".join([f"- {item}" for item in missing_elements]) if missing_elements else "None noted."
    formatted_memory = memory_context if memory_context else "No historical notes for this ticker."
    formatted_failure_reason = failure_reason if failure_reason else "N/A"

    user_message = OPTIMIZER_USER_PROMPT_TEMPLATE.format(
        current_draft=current_draft,
        failure_reason=formatted_failure_reason,
        overall_score=overall_score,
        feedback=feedback,
        missing_elements=formatted_missing,
        raw_evidence=raw_evidence[:25000],
        memory_context=formatted_memory
    )

    response = _optimizer_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": OPTIMIZER_SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        # Low temperature for accurate, grounded revisions
        temperature=0.3,
        max_tokens=3000
    )

    if not response.choices:
        raise RuntimeError(
            f"Optimizer returned no choices (model={model}). "
            "Check OpenRouter model settings and API response."
        )

    choice = response.choices[0]
    content = choice.message.content
    if not isinstance(content, str) or not content.strip():
        finish_reason = getattr(choice, "finish_reason", None)
        refusal = getattr(choice.message, "refusal", None)
        usage = getattr(response, "usage", None)
        raise RuntimeError(
            f"Optimizer returned no text (model={model}, finish_reason={finish_reason}, "
            f"refusal={refusal}, usage={usage}). "
            "If finish_reason='length', the model may have exhausted its output "
            "budget on reasoning. Try a higher max_tokens or a non-reasoning "
            "chat model. Check your OpenRouter credit limits first."
        )

    refined_draft = content.strip()
    return refined_draft

### Source implementation: `eval_opt_workflow.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [17]:
import datetime
from typing import Dict, Any, Tuple, List, Optional

# Run Evaluator-Optimizer workflow
def run_evaluator_optimizer_workflow(
    ticker: str,
    initial_draft: str,
    raw_evidence: str,
    max_iterations: int = 3,
    memory_store_path: str = "data/processed/memory_store.json"
) -> Tuple[str, EvaluationResult, List[Dict[str, Any]]]:
    """
    Orchestrates the Evaluator-Optimizer refinement loop and handles cross-run memory.
    
    Parameters:
        ticker (str): The target stock symbol (e.g., 'AAPL', 'NVDA').
        initial_draft (str): The raw draft produced by specialist research agents.
        raw_evidence (str): The combined daily evidence and news processing facts.
        max_iterations (int): Maximum number of evaluation-optimization passes (default: 3).
        memory_store_path (str): Path to persistent JSON memory file.
        
    Returns:
        Tuple containing:
            - final_draft (str): The final polished analysis report.
            - final_eval (EvaluationResult): The final evaluation rubric & score object.
            - history (List[Dict]): Detailed iteration history for auditing.
    """
    ticker = ticker.upper()
    memory_store = ResearchMemoryStore(filepath=memory_store_path)
    
    # Fetch historical run memory for ticker
    memory_context = memory_store.get_ticker_memory(ticker)
    print(f"[{ticker}] Loaded Memory Context:\n{memory_context}\n" + "-"*50)

    current_draft = initial_draft
    iteration_history: List[Dict[str, Any]] = []
    final_eval_result: Optional[EvaluationResult] = None

    # Evaluator-Optimizer Execution Loop
    for iteration in range(1, max_iterations + 1):
        print(f"[{ticker}] --- Starting Iteration {iteration}/{max_iterations} ---")
        
        # Evaluate current draft
        eval_result: EvaluationResult = evaluate_draft(
            current_draft=current_draft,
            raw_evidence=raw_evidence
        )
        final_eval_result = eval_result

        # Log iteration
        iteration_log = {
            "iteration": iteration,
            "draft": current_draft,
            "overall_score": eval_result.overall_score,
            "passed": eval_result.passed,
            "rubric": eval_result.rubric.model_dump(),
            "feedback": eval_result.refinement_feedback,
            "missing_elements": eval_result.missing_elements,
            "timestamp": datetime.datetime.now().isoformat()
        }
        iteration_history.append(iteration_log)

        print(f"[{ticker}] Iteration {iteration} Score: {eval_result.overall_score}/10 | Passed: {eval_result.passed} | Failure Reason: {eval_result.failure_reason}")

        # Check termination criteria
        if eval_result.passed or iteration == max_iterations:
            if eval_result.passed:
                print(f"[{ticker}] Quality threshold met on iteration {iteration}!")
            else:
                print(f"[{ticker}] Reached max iterations ({max_iterations}). Finalizing current draft.")
            break

        # Helps communication between evaluator and optimizer
        feedback_prompt = f"""
        EVALUATOR CRITIQUE:
        - Overall Score: {eval_result.overall_score}/10
        - Main Failure Reason: {eval_result.failure_reason}
        - Refinement Feedback: {eval_result.refinement_feedback}
        """
        
        # Refine draft using optimizer
        print(f"[{ticker}] Refining draft based on evaluator feedback...")
        current_draft = optimize_draft(
            current_draft=current_draft,
            feedback=feedback_prompt,
            raw_evidence=raw_evidence,
            memory_context=memory_context,
            missing_elements=eval_result.missing_elements
        )

    # Update memory storage with final results
    run_summary = (
        f"Final Score: {final_eval_result.overall_score}/10 after {len(iteration_history)} iterations. "
        f"Evaluator Notes: {final_eval_result.refinement_feedback[:150]}..."
    )
    memory_store.save_run(
        ticker=ticker,
        timestamp=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        score=final_eval_result.overall_score,
        summary=run_summary
    )
    print(f"[{ticker}] Persistent memory updated in {memory_store_path}.")

    return current_draft, final_eval_result, iteration_history

### Source implementation: `research_pipeline.py`

The code below is **executable notebook Python**, not a quoted source string or an imported local module.

In [18]:
"""Connect the planner-router orchestration to the evaluator-optimizer workflow.

Flow:
    orchestrate_research()            plan, route, run specialists, synthesize
    -> build_research_draft()         turn the results into a written draft
    -> build_raw_evidence()           collect the evidence the evaluator checks against
    -> run_evaluator_optimizer_workflow()   evaluate, refine, and save memory
"""

from __future__ import annotations

from pathlib import Path

import pandas as pd

MAX_NEWS_NOTES = 5


def _format_date(value) -> str:
    """Return a date as YYYY-MM-DD, or 'n/a' if missing."""
    if value is None or pd.isna(value):
        return "n/a"
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def _clean_text(value) -> str:
    """Return text with missing values replaced by an empty string."""
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def build_research_draft(result: dict) -> str:
    """Write an initial research draft from an orchestrate_research() result.

    Every figure in the draft comes directly from the specialist agents, so the
    evaluator can check it against the raw evidence.
    """

    ticker = result["ticker"]
    news = result["specialist_results"]["news_agent"]
    market = result["specialist_results"]["market_agent"]
    insider = result["specialist_results"]["insider_agent"]
    synthesis = result["final_synthesis"]

    lines = [f"# Investment Research Draft: {ticker}", ""]

    # Market performance
    lines.append("## Market Metrics")
    if market.get("row_count", 0) > 0:
        lines.extend(
            [
                f"- Trading days analyzed: {market['row_count']} "
                f"({_format_date(market['start_date'])} to {_format_date(market['end_date'])}) [Source: Alpha Vantage]",
                f"- Latest close: ${market['latest_close']:,.2f} "
                f"on volume of {market['latest_volume']:,.0f} shares [Source: Alpha Vantage]",
                f"- Average daily return: {market['average_daily_return']:.2%} [Source: Alpha Vantage]",
                f"- Average daily volume: {market['average_volume']:,.0f} shares [Source: Alpha Vantage]",
            ]
        )
    else:
        lines.append(market.get("message", "No market data available."))
    lines.append("")

    # Financials (required section in the optimizer prompt)
    lines.extend(
        [
            "## Financials & Revenue Performance",
            "Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded "
            "in the primary evidence dataset for this reporting period.",
            "",
        ]
    )

    # Insider activity
    lines.append("## SEC Form 4 Insider Activity")
    if insider.get("transaction_count", 0) > 0:
        lines.extend(
            [
                f"- Transactions: {insider['transaction_count']} by "
                f"{insider['unique_insiders']} insiders "
                f"({_format_date(insider['start_date'])} to {_format_date(insider['end_date'])})",
                f"- Acquisitions: {insider['acquired_transactions']}, "
                f"dispositions: {insider['disposed_transactions']} [Source: SEC Form 4]",
                f"- Total transaction value: ${insider['total_transaction_value']:,.2f} [Source: SEC Form 4]",
            ]
        )
    else:
        lines.append(insider.get("message", "No insider transactions available."))
    lines.append("")

    # News
    lines.append("## Financial News Analysis")
    if news.get("article_count", 0) > 0:
        categories = ", ".join(
            f"{name} ({count})" for name, count in news["categories"].items()
        )
        lines.append(f"- Articles analyzed: {news['article_count']} [Source: Marketaux]")
        lines.append(f"- Categories: {categories} [Source: Marketaux]")
        for note in news["research_notes"][:MAX_NEWS_NOTES]:
            title = _clean_text(note.get("title"))
            summary = _clean_text(note.get("summary")) or _clean_text(note.get("event"))
            category = _clean_text(note.get("category"))
            line = f"- {title} [{category}]"
            lines.append((f"{line}: {summary}" if summary else line) + " [Source: Marketaux]")
    else:
        lines.append(news.get("message", "No news available."))
    lines.append("")

    # Synthesis
    lines.append("## Risks")
    if synthesis["risks"]:
        lines.extend(f"- {risk} [Source: Alpha Vantage; SEC Form 4; Marketaux]" for risk in synthesis["risks"])
    else:
        lines.append("No risks identified from the available evidence.")
    lines.append("")

    lines.append("## Catalysts")
    if synthesis["catalysts"]:
        lines.extend(f"- {catalyst} [Source: Alpha Vantage; SEC Form 4; Marketaux]" for catalyst in synthesis["catalysts"])
    else:
        lines.append("No catalysts identified from the available evidence.")

    return "\n".join(lines)


def build_raw_evidence(ticker: str, project_root: str | Path = ".") -> str:
    """Build a bounded, traceable evidence summary for LLM evaluation/refinement.

    Keep full source datasets on disk; sending every CSV row to an LLM can
    exceed OpenRouter prompt limits. The report explicitly notes sampling.
    """
    ticker = ticker.upper().strip()
    processed = Path(project_root) / "data" / "processed"
    daily = pd.read_csv(processed / "daily_evidence.csv")
    news = pd.read_csv(processed / "news_research_results.csv")
    t_daily = daily[daily["ticker"].astype(str).str.upper() == ticker].copy()
    t_news = news[news["ticker"].astype(str).str.upper() == ticker].copy()

    if "date" in t_daily.columns:
        t_daily = t_daily.sort_values("date", ascending=False)
    if "published_at" in t_news.columns:
        t_news = t_news.sort_values("published_at", ascending=False)

    daily_cols = [col for col in [
        "ticker", "date", "open", "high", "low", "close", "volume",
        "daily_return", "news_count", "insider_transaction_count",
        "insider_total_value", "insider_names", "news_titles"
    ] if col in t_daily.columns]
    news_cols = [col for col in [
        "ticker", "published_at", "title", "category", "event", "summary",
        "financial_numbers", "source", "url"
    ] if col in t_news.columns]

    # Use short, inspectable samples while keeping all unabridged data on disk.
    daily_sample = t_daily[daily_cols].head(12).fillna("").copy()
    news_sample = t_news[news_cols].head(10).fillna("").copy()
    for frame in (daily_sample, news_sample):
        for col in frame.select_dtypes(include=["object", "str"]).columns:
            frame[col] = frame[col].astype(str).str.slice(0, 230)

    market_count = len(t_daily)
    news_count = len(t_news)
    daily_text = daily_sample.to_string(index=False) if market_count else "No daily evidence recorded."
    news_text = news_sample.to_string(index=False) if news_count else "No news research evidence recorded."
    text = (
        f"=== BOUNDED RESEARCH EVIDENCE FOR {ticker} ===\n"
        f"Daily evidence rows on disk: {market_count}; showing latest {len(daily_sample)}.\n"
        f"News research rows on disk: {news_count}; showing latest {len(news_sample)}.\n"
        "These are samples, not the complete datasets. Do not infer unshown facts.\n"
        f"\n=== DAILY MARKET & INSIDER EVIDENCE ===\n{daily_text}\n"
        f"\n=== PROCESSED NEWS EVIDENCE ===\n{news_text}\n"
    )
    return text[:25000] + ("\n[Evidence truncated to stay within the LLM prompt budget.]" if len(text) > 25000 else "")


def run_full_research(
    ticker: str,
    project_root: str | Path = ".",
    max_iterations: int = 3,
    memory_store_path: str | Path | None = None,
) -> dict:
    """Run the full system: orchestration, then evaluation and refinement with memory.

    Load the .env file before calling this function, because the evaluator and
    optimizer read the LLM settings when they are imported.
    """

    # Imported here so the orchestration part can run without LLM settings.

    project_root = Path(project_root)
    if memory_store_path is None:
        memory_store_path = project_root / "data" / "processed" / "memory_store.json"

    orchestration = orchestrate_research(ticker, project_root)
    initial_draft = build_research_draft(orchestration)
    raw_evidence = build_raw_evidence(ticker, project_root)

    final_report, final_evaluation, history = run_evaluator_optimizer_workflow(
        ticker=orchestration["ticker"],
        initial_draft=initial_draft,
        raw_evidence=raw_evidence,
        max_iterations=max_iterations,
        memory_store_path=str(memory_store_path),
    )

    return {
        "ticker": orchestration["ticker"],
        "orchestration": orchestration,
        "initial_draft": initial_draft,
        "raw_evidence": raw_evidence,
        "final_report": final_report,
        "final_evaluation": final_evaluation,
        "history": history,
    }


## B. Unified end-to-end workflow (original notebooks 00–06)

In [19]:
import os, sys, json, hashlib, ast, time
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
PROJECT_ROOT = Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / '.env', override=True)
set_seed(); ensure_directories()
TICKERS = get_target_tickers()
FORCE_REFRESH = os.getenv('FORCE_REFRESH', 'false').lower() in ('1','true','yes')
FORCE_NEWS_CHAIN = os.getenv('FORCE_NEWS_CHAIN', 'false').lower() in ('1','true','yes')
NEWS_CHAIN_LIMIT = int(os.getenv('NEWS_CHAIN_LIMIT', '10'))
MAX_ITERATIONS = int(os.getenv('MAX_ITERATIONS', '2'))
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')
LLM_API_KEY = os.getenv('LLM_API_KEY', '')
LLM_BASE_URL = os.getenv('LLM_BASE_URL', '').strip()
RAW_FILES = {'sec':PATHS['raw']/'sec_form4.csv', 'market':PATHS['raw']/'market_data.csv', 'news':PATHS['raw']/'news.csv'}
OUTPUT_FILES = {'sec': PATHS['processed']/'sec_form4_clean.csv','market':PATHS['processed']/'market_data_clean.csv','news':PATHS['processed']/'news_clean.csv'}
NEWS_INPUT = OUTPUT_FILES['news']; NEWS_OUTPUT = PATHS['processed']/'news_research_results.csv'
print('Root:', PROJECT_ROOT, '| Tickers:', TICKERS, '| Model:', LLM_MODEL)
print('Force refresh:', FORCE_REFRESH, '| News chain limit:', NEWS_CHAIN_LIMIT)


Root: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System | Tickers: ['AAPL'] | Model: openai/gpt-4.1-mini
Force refresh: False | News chain limit: 10


## 1. Ingest SEC Form 4, Alpha Vantage / yfinance, and Marketaux

Existing raw snapshots are loaded unless `FORCE_REFRESH=true`. Missing API credentials are only required for uncached sources.

In [20]:

def obtain_snapshot(name, fetch, empty_columns):
    path = RAW_FILES[name]
    if path.exists() and not FORCE_REFRESH:
        print(f'{name}: using cached {path}')
        return pd.read_csv(path)
    try:
        frame = fetch()
        if frame is None or frame.empty:
            print(f'WARNING: {name} returned no rows; writing empty schema')
            frame = pd.DataFrame(columns=empty_columns)
        frame.to_csv(path, index=False)
        print(f'{name}: saved {len(frame)} rows to {path}')
        return frame
    except Exception as exc:
        if path.exists():
            print(f'WARNING {name} refresh failed ({exc}); using previously saved snapshot')
            return pd.read_csv(path)
        raise RuntimeError(f'{name} ingestion failed and no cached snapshot is available: {exc}') from exc

def fetch_sec():
    ua = os.getenv('SEC_USER_AGENT', '')
    if '@' not in ua: raise ValueError('Set SEC_USER_AGENT in .env (must include contact email)')
    return fetch_form4_transactions(tickers=TICKERS, user_agent=ua, filings_per_ticker=int(os.getenv('SEC_FORM4_LIMIT','20')))

sec_raw = obtain_snapshot('sec', fetch_sec, OUTPUT_COLUMNS)

def fetch_market():
    key = os.getenv('ALPHA_VANTAGE_API_KEY', '')
    if key:
        try:
            df = fetch_daily_market_data(TICKERS, key)
            if not df.empty: return df
        except Exception as exc:
            print('Alpha Vantage unavailable; trying yfinance:', exc)
    import yfinance as yf
    frames=[]
    for ticker in TICKERS:
        h=yf.Ticker(ticker).history(period='3mo')
        if h.empty: continue
        h=h.reset_index();h['ticker']=ticker
        h=h.rename(columns={'Date':'date','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'})
        h['date']=pd.to_datetime(h['date']).dt.strftime('%Y-%m-%d')
        frames.append(h[['ticker','date','open','high','low','close','volume']])
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

market_raw=obtain_snapshot('market',fetch_market,['ticker','date','open','high','low','close','volume'])

def fetch_news():
    token=os.getenv('MARKETAUX_API_KEY','')
    if not token: raise ValueError('Set MARKETAUX_API_KEY in .env')
    return fetch_financial_news(TICKERS,token,limit_per_ticker=int(os.getenv('NEWS_LIMIT_PER_TICKER','9')),page_size=int(os.getenv('MARKETAUX_PAGE_SIZE','3')))

news_raw=obtain_snapshot('news',fetch_news,['ticker','published_at','title','description','content','source','url'])
display(pd.DataFrame([{'source':name,'rows':len(df),'snapshot':str(RAW_FILES[name])} for name,df in [('sec',sec_raw),('market',market_raw),('news',news_raw)]]))


sec: using cached C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\sec_form4.csv
market: using cached C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\market_data.csv
news: using cached C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\news.csv


,source,rows,snapshot
0,sec,61,C:\Users\admin\Desktop\AAI520\Investment-Resea...
1,market,100,C:\Users\admin\Desktop\AAI520\Investment-Resea...
2,news,50,C:\Users\admin\Desktop\AAI520\Investment-Resea...


## 2. Preprocess raw data

Reuses the normalization, relevance filtering, and quality rules from notebook 01. Persists three clean CSVs and a preprocessing manifest.

In [21]:
import hashlib
def normalize_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

def normalize_ticker(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )

def clean_text(series):
    return (
        series.astype("string")
        .str.replace(r"<[^>]+>", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def to_utc(series):
    return pd.to_datetime(series, errors="coerce", utc=True)

def report_rows(name, before, after):
    print(f"{name}: {before:,} -> {after:,} rows ({before-after:,} removed)")

def fingerprint(df):
    data = pd.util.hash_pandas_object(
        df.sort_index(axis=1).fillna("<NA>"),
        index=True
    ).values.tobytes()
    return hashlib.sha256(data).hexdigest()[:16]

def preprocess_sec(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "transaction_date", "owner_name"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"SEC data missing required columns: {missing}")

    optional = [
        "company_name", "owner_title", "filing_date",
        "transaction_code", "acquired_disposed",
        "shares", "price", "accession_number", "source_url"
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    for col in ["owner_name", "company_name", "owner_title"]:
        df[col] = clean_text(df[col])

    df["transaction_date"] = to_utc(df["transaction_date"])
    df["filing_date"] = to_utc(df["filing_date"])

    df["shares"] = pd.to_numeric(df["shares"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["transaction_value"] = df["shares"] * df["price"]

    df["transaction_code"] = df["transaction_code"].astype("string").str.strip().str.upper()
    df["acquired_disposed"] = df["acquired_disposed"].astype("string").str.strip().str.upper()

    df = df.dropna(subset=["ticker", "transaction_date", "owner_name"])
    df = df.drop_duplicates(
        subset=["accession_number", "ticker", "owner_name", "transaction_date",
                "transaction_code", "shares", "price"]
    )

    df = df.sort_values(["ticker", "transaction_date"]).reset_index(drop=True)
    report_rows("SEC Form 4", before, len(df))
    return df

def preprocess_market(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "date", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Market data missing required columns: {missing}")

    for col in ["open", "high", "low", "adj_close", "volume"]:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["date"] = to_utc(df["date"])

    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["ticker", "date", "close"])
    df = df.drop_duplicates(subset=["ticker", "date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    df["daily_return"] = df.groupby("ticker")["close"].pct_change()
    df["volume_change"] = df.groupby("ticker")["volume"].pct_change()

    if "adj_close" in df.columns and df["adj_close"].isna().all():
        df = df.drop(columns=["adj_close"])

    report_rows("Market", before, len(df))
    return df

COMPANY_ALIASES = {
    "AAPL": [
        "apple",
        "aapl",
    ],
}


def safe_lower(value):
    if pd.isna(value):
        return ""

    return str(value).lower()


def get_news_relevance_score(row):
    ticker = str(row.get("ticker", "")).upper()

    aliases = COMPANY_ALIASES.get(
        ticker,
        [ticker.lower()],
    )

    title = safe_lower(row.get("title"))

    article_text = " ".join(
        [
            title,
            safe_lower(row.get("description")),
            safe_lower(row.get("content")),
        ]
    )

    if any(alias in title for alias in aliases):
        return 2

    if any(alias in article_text for alias in aliases):
        return 1

    return 0

def preprocess_news(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["published_at", "title"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"News data missing required columns: {missing}")

    optional = [
        "ticker",
        "company_name",
        "description",
        "content",
        "source",
        "url",
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["published_at"] = to_utc(df["published_at"])

    for col in [
        "title",
        "description",
        "content",
        "company_name",
        "source",
    ]:
        df[col] = clean_text(df[col])

    with_url = df[df["url"].notna()].drop_duplicates(subset=["url"])
    without_url = df[df["url"].isna()].drop_duplicates(
        subset=["title", "published_at"]
    )
    df = pd.concat(
        [with_url, without_url],
        ignore_index=True,
    )

    df = df.dropna(
        subset=[
            "published_at",
            "title",
        ]
    )

    df["text"] = (
        df["title"].fillna("")
        + ". "
        + df["description"].fillna("")
        + " "
        + df["content"].fillna("")
    )
    df["text"] = clean_text(df["text"])

    before_relevance = len(df)

    df["relevance_score"] = df.apply(
        get_news_relevance_score,
        axis=1,
    )

    print("\nNews relevance scores before filtering:")
    print(
        df["relevance_score"]
        .value_counts()
        .sort_index()
    )

    df = df[
        df["relevance_score"] >= 1
    ].copy()

    print(
        "News relevance filter:",
        f"{before_relevance:,} -> {len(df):,} rows "
        f"({before_relevance - len(df):,} removed)",
    )

    df = (
        df.sort_values(
            "published_at",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    report_rows(
        "News",
        before,
        len(df),
    )

    return df


In [22]:
processed = {
 'sec':preprocess_sec(sec_raw),
 'market':preprocess_market(market_raw),
 'news':preprocess_news(news_raw),
}
if processed['market'].empty:
    raise ValueError('No usable market rows after preprocessing; load market data before research.')
for name, frame in processed.items():
    frame.to_csv(OUTPUT_FILES[name],index=False)
    print('Saved', OUTPUT_FILES[name],len(frame),'rows')
manifest={'seed':SEED,'datasets':{k:{'rows':len(v),'columns':list(v.columns),'fingerprint':fingerprint(v),'output_file':str(OUTPUT_FILES[k])} for k,v in processed.items()}}
(PATHS['processed']/'preprocessing_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.DataFrame([{'dataset':k,'rows':len(v),'columns':len(v.columns)} for k,v in processed.items()]))


SEC Form 4: 61 -> 57 rows (4 removed)
Market: 100 -> 100 rows (0 removed)

News relevance scores before filtering:
relevance_score
0    24
1     8
2    16
Name: count, dtype: int64
News relevance filter: 48 -> 24 rows (24 removed)
News: 50 -> 24 rows (26 removed)
Saved C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\sec_form4_clean.csv 57 rows
Saved C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\market_data_clean.csv 100 rows
Saved C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_clean.csv 24 rows


,dataset,rows,columns
0,sec,57,13
1,market,100,9
2,news,24,10


## 3. Assemble daily evidence

Uses the included `assemble_daily_evidence` implementation to align market, SEC, and news observations on ticker/date.

In [23]:
evidence=assemble_daily_evidence(processed['market'],processed['news'],processed['sec'])
assert not evidence.duplicated(['ticker','date']).any()
evidence.to_csv(PATHS['processed']/'daily_evidence.csv',index=False)
print('Daily evidence:',len(evidence),'rows')
display(evidence.head(10))

Daily evidence: 108 rows


,ticker,date,open,high,low,close,volume,daily_return,volume_change,news_count,news_titles,news_urls,insider_transaction_count,insider_total_value,insider_names
0,AAPL,2026-02-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,1.0,0.00,WAGNER SUSAN
1,AAPL,2026-03-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,2.0,8135903.36,Newstead Jennifer
2,AAPL,2026-04-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,15.0,34320628.17,O'BRIEN DEIRDRE || Khan Sabih || COOK TIMOTHY D
3,AAPL,2026-04-02 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,8.0,24173072.77,O'BRIEN DEIRDRE || COOK TIMOTHY D
4,AAPL,2026-04-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,10.0,1514654.55,Borders Ben || Parekh Kevan
5,AAPL,2026-04-23 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,1.0,421850.00,Parekh Kevan
6,AAPL,2026-04-28 00:00:00+00:00,272.335,273.23,268.66,270.71,40018940.0,NaN,NaN,0,<NA>,NaN,NaN,NaN,<NA>
7,AAPL,2026-04-29 00:00:00+00:00,267.550,271.04,267.04,270.17,30047869.0,-0.001995,-0.249159,0,<NA>,NaN,NaN,NaN,<NA>
8,AAPL,2026-04-30 00:00:00+00:00,270.500,276.00,268.14,271.35,91848230.0,0.004368,2.056730,0,<NA>,NaN,NaN,NaN,<NA>
9,AAPL,2026-05-01 00:00:00+00:00,278.855,287.22,278.37,280.14,79915442.0,0.032394,-0.129919,0,<NA>,NaN,NaN,NaN,<NA>


## 4. Prompt chaining: classify → extract → summarize news

This stage uses an OpenAI-compatible LLM. Previously processed URLs are reused unless `FORCE_NEWS_CHAIN=true`. If there is no relevant news, a schema-only output is created for downstream agents.

In [24]:
from openai import OpenAI
NEWS_CATEGORIES=['earnings','product_service','management','regulation_legal','merger_acquisition','analyst_investor','macro_market','other']
NEWS_COLUMNS=['ticker','published_at','title','source','url','relevance_score','category','classification_reason','event','key_facts','people','organizations','financial_numbers','summary']
news=processed['news'].sort_values(['relevance_score','published_at'],ascending=[False,False]).copy()
client=None

def call_llm_json(system_prompt,user_prompt,temperature=0):
    global client
    if not LLM_API_KEY: raise ValueError('LLM_API_KEY is required to process new articles')
    if client is None:
        client=OpenAI(api_key=LLM_API_KEY,**({'base_url':LLM_BASE_URL} if LLM_BASE_URL else {}))
    resp=client.chat.completions.create(model=LLM_MODEL,temperature=temperature,max_tokens=1200,messages=[{'role':'system','content':system_prompt},{'role':'user','content':user_prompt}])
    raw=resp.choices[0].message.content
    if not raw: raise ValueError('Empty LLM response')
    txt=raw.strip()
    if txt.startswith('```'): txt=txt.strip('`').removeprefix('json').strip()
    try:return json.loads(txt)
    except json.JSONDecodeError:
        parsed=ast.literal_eval(txt)
        if isinstance(parsed,dict): return parsed
        raise ValueError('LLM response is not a JSON object')


In [25]:
def classify_article(article_text):
    categories = ", ".join(NEWS_CATEGORIES)

    system_prompt = """
You classify financial news for an investment research pipeline.

Use only the article provided by the user.
Do not make investment recommendations.
Return valid JSON only.
""".strip()

    user_prompt = f"""
Classify this article into exactly one of these categories:

{categories}

Category definitions:
- earnings: earnings reports, revenue, profit, guidance, or financial results
- product_service: products, services, launches, technology, or operations
- management: executives, boards, leadership, hiring, or departures
- regulation_legal: lawsuits, regulation, investigations, or government action
- merger_acquisition: mergers, acquisitions, divestitures, or major strategic deals
- analyst_investor: analyst ratings, price targets, investor commentary, or capital markets
- macro_market: broader economic, sector, commodity, interest-rate, or market developments
- other: relevant company news that does not fit the categories above

Return exactly this JSON structure:

{{
  "category": "one_category_from_the_list",
  "classification_reason": "one concise sentence"
}}

Article:
{article_text}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    category = result.get("category")

    if category not in NEWS_CATEGORIES:
        raise ValueError(
            f"Invalid category returned by model: {category}"
        )

    return result


In [26]:
def extract_article_facts(
    article_text,
    category,
):
    system_prompt = """
You extract structured evidence from financial news.

Use only facts explicitly supported by the supplied article.
Do not infer missing facts.
Do not make investment recommendations.
Return valid JSON only.
""".strip()

    user_prompt = f"""
The article was classified as:

{category}

Extract the important research evidence from the article.

Return exactly this JSON structure:

{{
  "event": "one-sentence description of the primary event",
  "key_facts": [
    "fact 1",
    "fact 2"
  ],
  "people": [
    "person name"
  ],
  "organizations": [
    "organization name"
  ],
  "financial_numbers": [
    "financial or quantitative value exactly as stated"
  ]
}}

Use empty lists when a field has no supported information.

Article:
{article_text}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    expected_list_fields = [
        "key_facts",
        "people",
        "organizations",
        "financial_numbers",
    ]

    for field in expected_list_fields:
        value = result.get(field)

        if value is None:
            result[field] = []

        elif not isinstance(value, list):
            result[field] = [str(value)]

    if not result.get("event"):
        result["event"] = ""

    return result


In [27]:
def summarize_article(
    category,
    extracted_facts,
):
    system_prompt = """
You write concise evidence-grounded financial research summaries.

Use only the structured evidence supplied by the user.
Do not introduce new facts.
Do not make a buy, sell, or hold recommendation.
Return valid JSON only.
""".strip()

    evidence_json = json.dumps(
        extracted_facts,
        ensure_ascii=False,
        indent=2,
    )

    user_prompt = f"""
Article category:

{category}

Structured evidence:

{evidence_json}

Write a concise research summary that:
1. identifies the main event,
2. includes the most important supported facts,
3. briefly explains why the event may matter for company research,
4. avoids unsupported conclusions or investment recommendations.

Return exactly:

{{
  "summary": "2-4 sentence research summary"
}}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    return result


In [28]:
def run_news_chain(article):
    classification = classify_article(
        article["text"]
    )

    extraction = extract_article_facts(
        article_text=article["text"],
        category=classification["category"],
    )

    summary_result = summarize_article(
        category=classification["category"],
        extracted_facts=extraction,
    )

    return {
        "ticker": article["ticker"],
        "published_at": article["published_at"],
        "title": article["title"],
        "source": article.get("source"),
        "url": article.get("url"),
        "relevance_score": article.get("relevance_score"),
        "category": classification["category"],
        "classification_reason": classification.get(
            "classification_reason",
            "",
        ),
        "event": extraction.get("event", ""),
        "key_facts": extraction.get("key_facts", []),
        "people": extraction.get("people", []),
        "organizations": extraction.get(
            "organizations",
            [],
        ),
        "financial_numbers": extraction.get(
            "financial_numbers",
            [],
        ),
        "summary": summary_result.get(
            "summary",
            "",
        ),
    }


In [29]:
existing_results=pd.DataFrame(columns=NEWS_COLUMNS)
if NEWS_OUTPUT.exists() and not FORCE_NEWS_CHAIN:
    existing_results=pd.read_csv(NEWS_OUTPUT)
completed=set(existing_results['url'].dropna().astype(str)) if 'url' in existing_results else set()
todo=news[~news['url'].astype(str).isin(completed)] if completed else news
todo=todo.head(NEWS_CHAIN_LIMIT)
print('Articles to process:',len(todo),'| Cached:',len(existing_results))
new_results=[]
for _,article in todo.iterrows():
    try:
        print('Processing:',article['ticker'],str(article['title'])[:70])
        new_results.append(run_news_chain(article))
    except Exception as exc:
        print('WARNING: article processing failed:',type(exc).__name__,str(exc))
results=pd.concat([existing_results,pd.DataFrame(new_results)],ignore_index=True)
if 'url' in results: results=results.drop_duplicates(['url'],keep='last')
for col in NEWS_COLUMNS:
    if col not in results: results[col]=pd.NA
results=results[NEWS_COLUMNS]
results.to_csv(NEWS_OUTPUT,index=False)
print('Saved',NEWS_OUTPUT,'(',len(results),'articles)')
display(results[['ticker','title','category','summary']].tail(10))


Articles to process: 0 | Cached: 24
Saved C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_research_results.csv ( 24 articles)


,ticker,title,category,summary
14,AAPL,OpenAI Fires Back at Apple (AAPL),regulation_legal,OpenAI denied Apple's allegations of trade sec...
15,AAPL,AAPL Looks 16.2% Overvalued on GF Value™,analyst_investor,An article on Apple Inc. (AAPL) describes the ...
16,AAPL,Mag 7 Laggard Meta Has Quietly Turned Positive...,macro_market,Meta Platforms has quietly turned positive for...
17,AAPL,The Same Stock but in 2 Different IRAs: By Ret...,regulation_legal,The article explains that Apple shares held in...
18,AAPL,Wall Street Brunch: U.S.-China Summit In Spotl...,earnings,The main event is the U.S.-China summit where ...
19,AAPL,Day 1 sales of iPhone 18 Pro 15-28% higher tha...,product_service,Apple launched sales of the iPhone 18 Pro on S...
20,AAPL,Wall Street Breakfast With Steven Cress (undef...,macro_market,The main event is a witching day on which Stev...
21,AAPL,Wall Street Breakfast With Steven Cress (undef...,analyst_investor,Generac's stock surged after announcing an agr...
22,AAPL,Tech power players land seat at table for high...,management,Donald Trump plans a state dinner next week fo...
23,AAPL,Wall Street Breakfast Podcast: Amazon Supercha...,product_service,Generac (GNRC) lands an $8B opportunity with A...


## 5. Planner → router → specialist agents → synthesis

The planner chooses steps based on processed data, the router delegates to market/news/insider specialists, and synthesis collects risks and catalysts.

In [30]:
TICKER = TICKERS[0]
all_orchestrations={}
for ticker in TICKERS:
    run=orchestrate_research(ticker,PROJECT_ROOT)
    all_orchestrations[ticker]=run
    print(f"{ticker}: agents={run['agents_called']}; {run['final_synthesis']['summary']}")
research_plan=all_orchestrations[TICKER]
display_orchestration(research_plan)
display(pd.DataFrame(research_plan['routing_decisions']))


AAPL: agents=['insider_agent', 'market_agent', 'news_agent']; {'news_articles': 24, 'market_rows': 100, 'insider_transactions': 57}
Ticker: AAPL
Objective: Conduct structured multi-agent investment research for AAPL.
Planner decisions:
- news_analysis: include (Found 24 processed news articles.)
- market_analysis: include (Found 100 trading days of market data.)
- insider_analysis: include (Found 57 SEC Form 4 insider transactions.)
Step 1: news_analysis -> news_agent
Step 2: market_analysis -> market_agent
Step 3: insider_analysis -> insider_agent
Step 4: risk_analysis -> synthesis_agent
Step 5: catalyst_analysis -> synthesis_agent
Step 6: final_synthesis -> synthesis_agent
Agents called: insider_agent, market_agent, news_agent
Final synthesis
Risks:
- Macro or broader market developments may affect the stock.
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data.
Catalysts:
- Recent analyst or investor commentary may influence market expectation

,step_id,task,agent,reason,description
0,1,news_analysis,news_agent,This task depends primarily on processed finan...,Review recent financial news and macro develop...
1,2,market_analysis,market_agent,"This task depends primarily on market price, r...","Analyze recent price, return, and trading-volu..."
2,3,insider_analysis,insider_agent,This task depends primarily on SEC Form 4 insi...,Review recent SEC Form 4 insider transactions ...
3,4,risk_analysis,synthesis_agent,This task requires combining evidence from mul...,Identify important risks using the available e...
4,5,catalyst_analysis,synthesis_agent,This task requires combining evidence from mul...,Identify possible catalysts or notable events ...
5,6,final_synthesis,synthesis_agent,This task requires combining evidence from mul...,Synthesize the research evidence into a concis...


## 6. Draft generation and evidence grounding

Build a draft from the specialist outputs, with mandatory evaluator section headings and source tags. Compare against the saved raw-evidence text.

In [31]:
initial_draft=build_research_draft(research_plan)
raw_evidence=build_raw_evidence(TICKER,PROJECT_ROOT)
print('Raw evidence characters:',len(raw_evidence))
display(Markdown(initial_draft))


Raw evidence characters: 14607


# Investment Research Draft: AAPL

## Market Metrics
- Trading days analyzed: 100 (2026-04-28 to 2026-09-18) [Source: Alpha Vantage]
- Latest close: $336.13 on volume of 86,588,203 shares [Source: Alpha Vantage]
- Average daily return: 0.24% [Source: Alpha Vantage]
- Average daily volume: 52,609,596 shares [Source: Alpha Vantage]

## Financials & Revenue Performance
Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period.

## SEC Form 4 Insider Activity
- Transactions: 57 by 9 insiders (2026-02-24 to 2026-09-15)
- Acquisitions: 11, dispositions: 46 [Source: SEC Form 4]
- Total transaction value: $168,657,491.72 [Source: SEC Form 4]

## Financial News Analysis
- Articles analyzed: 24 [Source: Marketaux]
- Categories: analyst_investor (10), product_service (6), regulation_legal (3), earnings (2), macro_market (2), management (1) [Source: Marketaux]
- Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club. [product_service]: The main event is a prediction that Apple's foldable iPhone Duo, priced at $1,999, will be a hit and will see Apple join NVIDIA in the $5 trillion club. Key facts include the $1,999 price, the forecast of success, Apple's NASDAQ listing as AAPL, and the involvement of Apple and NVIDIA. If realized, the event could indicate a strong demand signal for Apple’s foldable device and a milestone implied by a $5 trillion club, informing company research on product strategy and competitive positioning with NVIDIA. [Source: Marketaux]
- AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists [analyst_investor]: Apple Inc. is reportedly 17.3% overvalued on GF Value™ as insider selling persists. The key facts are that AAPL is shown as 17.3% overvalued on GF Value™ and insider selling persists for Apple. This information may be relevant for company research as it highlights valuation concerns and ongoing insider activity that could influence investor sentiment and governance risk. [Source: Marketaux]
- Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro [product_service]: Apple unveiled the A20 Pro chip, Ceramic Shield in the iPhone 18 Pro, and an innovative folding screen on September 20, 2026. The key facts include the launch date, the new A20 Pro chip, Ceramic Shield integration in the iPhone 18 Pro, and the folding screen feature. This event may matter for company research as it signals potential changes in product capabilities, competitive positioning, and research and development priorities for Apple. [Source: Marketaux]
- John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy? [analyst_investor]: Bank of America cut Apple's price target after John Ternus's first iPhone launch event as Apple's CEO, where Apple unveiled the iPhone 18 Pro and Pro Max. The cut was prompted by a memory processor crunch. These developments illustrate how a major bank's price target may respond to a leadership-led product launch and potential component constraints, providing context for company research without offering an investment recommendation. [Source: Marketaux]
- Apple Raised Stakes with its $2000-Priced Foldable. But Can it Be the New Growth Engine? [product_service]: Apple released its first foldable phone, the Duo, priced at $2000. The Duo is Apple's first foldable phone and represents the most significant iPhone design change in recent years. For company research, this event may matter as it signals a major design shift and the introduction of a foldable product, which could affect analysis of Apple's product strategy and pricing. [Source: Marketaux]

## Risks
- Macro or broader market developments may affect the stock. [Source: Alpha Vantage; SEC Form 4; Marketaux]
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data. [Source: Alpha Vantage; SEC Form 4; Marketaux]

## Catalysts
- Recent analyst or investor commentary may influence market expectations. [Source: Alpha Vantage; SEC Form 4; Marketaux]
- Average daily market return over the available period is positive. [Source: Alpha Vantage; SEC Form 4; Marketaux]
- The available Form 4 data includes insider acquisition transactions. [Source: Alpha Vantage; SEC Form 4; Marketaux]

## 7. Evaluator → optimizer → memory

Run the full research workflow using the included evaluator and optimizer implementations. For each ticker: evaluate the initial draft, refine until passing or maximum iterations, and write persistent evaluation history. Requires `LLM_API_KEY`; evaluator requests structured output support from the selected provider.

In [32]:
if not LLM_API_KEY:
    raise ValueError('Set LLM_API_KEY in .env to execute the evaluator–optimizer workflow.')
MEMORY_PATH=PATHS['processed']/'memory_store.json'
research_runs={}
for ticker in TICKERS:
    print('\n'+'='*24+' '+ticker+' '+'='*24)
    try:
        research_runs[ticker]=run_full_research(ticker,PROJECT_ROOT,max_iterations=MAX_ITERATIONS,memory_store_path=MEMORY_PATH)
    except Exception as exc:
        print(f'{ticker}: research workflow failed: {type(exc).__name__}: {exc}')
        raise



======================== AAPL ========================
[AAPL] Loaded Memory Context:
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-04 12:09:38] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 2 iterations. Evaluator Notes: Provide a compact, fully tabulated insider-ledger tied to the 57 trades and $168,657,491.72 total (line-by-line: insider, date, trade type, shares, va...
Run #2 [2026-10-09 11:36:25] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 2 iterations. Evaluator Notes: The draft is well-structured with clear headers for Market Metrics, SEC Insider Activity, and News, satisfying the mandatory sections requirement. The...
--------------------------------------------------
[AAPL] --- Starting Iteration 1/2 ---
[AAPL] Iteration 1 Score: 4/10 | Passed: False | Failure Reason: Hard gate failure: EVERY individual bullet point containing factual metrics MUST include an inline source tag (e.g., [Source: Alpha Vantage], [Source: SEC Form 4], [Source

In [33]:
for ticker, research in research_runs.items():
    history=pd.DataFrame([{'iteration':step['iteration'],'score':step['overall_score'],'passed':step['passed'],'feedback':step['feedback']} for step in research['history']])
    print('\n',ticker,'— final score:',research['final_evaluation'].overall_score,'| passed:',research['final_evaluation'].passed)
    display(history)
    display(Markdown(research['final_report']))
    (PATHS['processed']/f'{ticker}_final_report.md').write_text(research['final_report'],encoding='utf-8')
    history.to_csv(PATHS['processed']/f'{ticker}_evaluation_history.csv',index=False)
print('\nMemory file:',MEMORY_PATH)



 AAPL — final score: 9 | passed: True


,iteration,score,passed,feedback
0,1,4,False,Please fix the following issues: EVERY individ...
1,2,9,True,The draft is well-structured with clear header...


# Investment Research Draft: AAPL

## Market Metrics
- Trading days analyzed: 100 (2026-04-28 to 2026-09-18) [Source: Alpha Vantage]
- Latest close: $336.13 on volume of 86,588,203 shares [Source: Alpha Vantage]
- Average daily return: 0.24% [Source: Alpha Vantage]
- Average daily volume: 52,609,596 shares [Source: Alpha Vantage]

## Financials & Revenue Performance
- Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period.

## SEC Form 4 Insider Activity
- Transactions: 57 by 9 insiders from 2026-02-24 to 2026-09-15 [Source: SEC Form 4]
- Acquisitions: 11, dispositions: 46 [Source: SEC Form 4]
- Total transaction value: $168,657,491.72 [Source: SEC Form 4]

## Financial News Analysis
- Articles analyzed: 24 [Source: Marketaux]
- Categories: analyst_investor (10), product_service (6), regulation_legal (3), earnings (2), macro_market (2), management (1) [Source: Marketaux]
- Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club. The main event is a prediction that Apple's foldable iPhone Duo, priced at $1,999, will be a hit and will see Apple join NVIDIA in the $5 trillion club. Key facts include the $1,999 price, the forecast of success, Apple's NASDAQ listing as AAPL, and the involvement of Apple and NVIDIA. If realized, the event could indicate a strong demand signal for Apple’s foldable device and a milestone implied by a $5 trillion club, informing company research on product strategy and competitive positioning with NVIDIA. [Source: Marketaux]
- AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists. Apple Inc. is reportedly 17.3% overvalued on GF Value™ as insider selling persists. The key facts are that AAPL is shown as 17.3% overvalued on GF Value™ and insider selling persists for Apple. This information may be relevant for company research as it highlights valuation concerns and ongoing insider activity that could influence investor sentiment and governance risk. [Source: Marketaux]
- Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro. Apple unveiled the A20 Pro chip, Ceramic Shield in the iPhone 18 Pro, and an innovative folding screen on September 20, 2026. The key facts include the launch date, the new A20 Pro chip, Ceramic Shield integration in the iPhone 18 Pro, and the folding screen feature. This event may matter for company research as it signals potential changes in product capabilities, competitive positioning, and research and development priorities for Apple. [Source: Marketaux]
- John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Bank of America cut Apple's price target after John Ternus's first iPhone launch event as Apple's CEO, where Apple unveiled the iPhone 18 Pro and Pro Max. The cut was prompted by a memory processor crunch. These developments illustrate how a major bank's price target may respond to a leadership-led product launch and potential component constraints, providing context for company research without offering an investment recommendation. [Source: Marketaux]
- Apple Raised Stakes with its $2000-Priced Foldable. Apple released its first foldable phone, the Duo, priced at $2000. The Duo is Apple's first foldable phone and represents the most significant iPhone design change in recent years. For company research, this event may matter as it signals a major design shift and the introduction of a foldable product, which could affect analysis of Apple's product strategy and pricing. [Source: Marketaux]

## Risks
- Macro or broader market developments may affect the stock. [Source: Alpha Vantage] [Source: SEC Form 4] [Source: Marketaux]
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data. [Source: SEC Form 4]

## Catalysts
- Recent analyst or investor commentary may influence market expectations. [Source: Marketaux]
- Average daily market return over the available period is positive. [Source: Alpha Vantage]
- The available Form 4 data includes insider acquisition transactions. [Source: SEC Form 4]


Memory file: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\memory_store.json


## 8. Outputs and reproducibility

Outputs: `data/raw/{sec_form4,market_data,news}.csv`, `data/processed/{sec_form4_clean,market_data_clean,news_clean,daily_evidence,news_research_results}.csv`, `preprocessing_manifest.json`, `memory_store.json`, plus per-ticker final Markdown reports and evaluation CSVs. Re-running with cached files will save API calls. Set `FORCE_REFRESH=true` or `FORCE_NEWS_CHAIN=true` only when intentionally refreshing those stages.
